# Lab 1
# Open Web Standards — Publishing & Consuming a WMS/WFS with GeoServer
### GISS/GEOG 366/368 · Web Mapping & Web GIS

**Unit 2 Focus:** OGC web service standards (WMS, WFS, WMTS) · GeoServer as a self-hosted OGC implementation · Interoperability between open-source and proprietary stacks

**This is your first graded lab** (Lab Exercises, Performance Task 1) and it also opens **Project Topic Ideation A** — the first of three short checkpoints that build toward your Week 11 final project.

---

## Where We're Working: Silver City & Southwestern New Mexico

We're staying with the same shared region from Lab 0 — Silver City, Grant County, and the Gila National Forest — pulled from the **NM RGIS Clearinghouse** (https://rgis.unm.edu/). One shared place means we can compare notes on what published, and what broke, before you strike out on your own place for the final project.

## Learning Goals

By the end of this lab, you will be able to:
- Explain why an open standard like WMS/WFS lets a self-hosted server and a proprietary platform interoperate *(K02)*
- Stand up a local GeoServer instance and publish a real dataset as a WMS **and** a WFS layer *(S02)*
- Consume that published layer from an open-source client *and* attempt to consume it from ArcGIS Online, and explain what does (or doesn't) work about each *(K01, K02)*
- Read a `GetCapabilities` document and connect it to what you saw in lecture *(K02)*
- Take your first concrete step on the semester's final project: naming a real place and audience *(PK2 — Topic Ideation A)*

## How This Notebook Is Organized

Same as Lab 0 — the **Geo-Inquiry Process**: **Ask → Collect → Visualize → Create → Act.**

## A Note on Today's Two Deliverables

Today you're doing two different things that happen to share a notebook: (1) the graded lab itself — publishing and consuming a WMS/WFS — and (2) a short, low-stakes **Topic Ideation A** checkpoint, where you start naming a real place and audience for your Week 11 final project. Topic Ideation A isn't graded on the quality of your idea — it exists so that you're not starting your final project topic from zero in Week 11. You'll revisit and refine it in Topic Ideation B (Week 5) and C (Week 9).

## Before You Begin

| Tool | Cost / Access | Install? | Role this semester |
|---|---|---|---|
| **GeoServer** | Free, open-source | Yes — see Collect below for two install options | Open-source pathway: self-hosted OGC services, all semester |
| **A sample dataset** | Free | Download from NM RGIS Clearinghouse | The data you'll publish today |
| **ArcGIS Online** | Provided via WNMU organizational account | None | Consuming your published service, proprietary side |
| **QGIS** *(optional but recommended)* | Free | https://qgis.org/download/ | Handy for inspecting/reprojecting data before you publish it |
| **A modern browser with Dev Tools** | Free | Already set up in Lab 0 | Reading `GetCapabilities` and watching WMS/WFS requests |

**Links**
- GeoServer download: https://geoserver.org/download/
- GeoServer Docker image (alternative install path): https://hub.docker.com/r/kartoza/geoserver
- NM RGIS Clearinghouse: https://rgis.unm.edu/
- GeoServer public demo (for reference, not what you'll publish to): https://data.geoserver.org/geoserver/web/

> **[INSTRUCTOR NOTE — confirm before class]** If the department has a shared, always-on GeoServer instance instead of (or in addition to) a local install, replace the install instructions below with that server's URL and student credentials. A locally-installed GeoServer only exists on your own machine — the "Consume it from ArcGIS Online" step later in this lab depends on whether your GeoServer is reachable from the open internet, which a local install is not by default. See the callout in Step 4 for how to handle this.

## Step 1: Ask

Before installing anything, sit with a couple of questions from lecture.

> **EQ2 — What happens elsewhere in this system when I change one part of it — a tile size, a data format, a query, an interface choice — and who ends up reached, or excluded, on the other end?**

Take a minute and jot down your first-pass answers (edit the cell below):
- If you publish the *same* dataset as both a WMS and a WFS, what does each version let a user *do* that the other doesn't?
- GeoServer is free to install, but it has to run on a computer that's on and reachable. ArcGIS Online costs a license, but WNMU is always running it for you. What does that tradeoff mean for who can realistically stand up a live geospatial service?
- Pick (even loosely) a place and an audience you might want to build your final project around. It's okay if this changes completely by Week 5.

**Your answers (double-click this cell to edit):**

-
-
-

---

## Step 2: Collect

Get GeoServer running and grab a dataset to publish.

<details>
<summary><b>Open-source pathway — install & run GeoServer: click to expand</b></summary>

Pick **one** of these two paths — you only need GeoServer running, however you get there.

**Option A — Native installer**
1. Download the platform-specific installer/binary from https://geoserver.org/download/ (the "Platform Independent Binary" works everywhere if you have Java installed; installers exist for Windows/macOS).
2. Start GeoServer (on the binary, run `bin/startup.sh` or `bin/startup.bat`).
3. In your browser, go to `http://localhost:8080/geoserver`. You should see the GeoServer welcome page.
4. Log in with the default credentials (`admin` / `geoserver`) — you'll see this prompt in the top-right of the welcome page.

**Option B — Docker**
1. Install Docker Desktop if you don't already have it: https://www.docker.com/products/docker-desktop/
2. Run: `docker run -p 8080:8080 kartoza/geoserver`
3. Once it finishes starting up, go to `http://localhost:8080/geoserver` and log in as above.

</details>

<details>
<summary><b>Get a dataset — NM RGIS Clearinghouse: click to expand</b></summary>

1. Go to https://rgis.unm.edu/ and search for a small vector dataset covering Grant County or the Gila region — trails, fire perimeters, and public land boundaries are good, manageable choices.
2. Download it as a **shapefile** (`.shp` + its sidecar files, usually zipped together).
3. Unzip it somewhere you can find it — you'll upload this into GeoServer in Step 4.
4. *(Optional)* Open it in QGIS first just to confirm it looks right and note its coordinate reference system (CRS) — you'll need to know this when you publish it.

</details>

📸 **Save a screenshot now** — the GeoServer welcome page after logging in. Label it *Screenshot 1*.

---

## Step 3: Visualize

Before publishing anything of your own, get oriented in GeoServer's interface and see what a `GetCapabilities` document actually looks like.

<details>
<summary><b>Explore GeoServer's built-in demo layers: click to expand</b></summary>

1. From the GeoServer welcome page, click **Layer Preview** in the left sidebar. GeoServer ships with a handful of sample layers (states, populated places, etc.) already published — this is a fast way to see the publish workflow's *output* before you've built one yourself.
2. Click **OpenLayers** next to any sample layer — this opens an interactive preview, similar to what your own layer will look like once published.
3. Now find that same layer's **WMS `GetCapabilities` URL**. Click **Layer Preview**, then look for the small format links, or go directly to `http://localhost:8080/geoserver/wms?service=WMS&version=1.3.0&request=GetCapabilities`. Open it in a new tab.
4. Skim the raw XML. Find: the list of `<Layer>` entries (this is the "menu" from lecture), and the supported `<Format>` types for `GetMap`.
5. Now edit that same URL, changing `GetCapabilities` to `GetMap` with a specific layer and bounding box (GeoServer's Layer Preview page generates one of these for you automatically — click **Common Formats → PNG** to see it in action). Compare: one URL gave you a menu, the other gave you a picture.

</details>

📸 **Save a screenshot now** — the raw `GetCapabilities` XML with at least one `<Layer>` entry visible. Label it *Screenshot 2*.

**What you noticed (double-click to edit):**
- How many layers appeared in the `GetCapabilities` document?
- What image formats does `GetMap` support, based on what you saw?

---

## Step 4: Create

Now publish your own dataset, then try consuming it from both pathways.

<details>
<summary><b>Open-source pathway — publish your dataset in GeoServer: click to expand</b></summary>

1. In GeoServer, go to **Workspaces** → **Add new workspace**. Give it a short name (e.g., `giss366`) and a namespace URI (any URL-shaped string works, e.g., `http://giss366.wnmu.edu`).
2. Go to **Stores** → **Add new Store** → **Shapefile**. Point it at the shapefile you downloaded in Step 2, assign it to your new workspace, and save.
3. GeoServer will detect the layer inside your store and prompt you to **publish** it. Click **Publish**.
4. On the layer configuration page:
   - Under **Data**, set the **Declared SRS** to match your data's CRS (QGIS told you this in Step 2, if you checked) and click **Compute from data** for the bounding boxes.
   - Under **Publishing**, note the default style applied — you can leave it, or explore GeoServer's built-in styles for something more legible (e.g., a line style if your data is trails).
5. Save. Go back to **Layer Preview**, find your new layer, and open it in **OpenLayers** to confirm it renders.
6. Your layer is now live as **both** WMS and WFS, automatically. Find both endpoints:
   - WMS `GetCapabilities`: `http://localhost:8080/geoserver/[your-workspace]/wms?service=WMS&version=1.3.0&request=GetCapabilities`
   - WFS `GetCapabilities`: `http://localhost:8080/geoserver/[your-workspace]/wfs?service=WFS&version=2.0.0&request=GetCapabilities`

</details>

📸 **Save a screenshot now** — your published layer rendering in GeoServer's OpenLayers preview. Label it *Screenshot 3*.

<details>
<summary><b>Consume your WFS from Python: click to expand</b></summary>

Run the code cell below (edit the URL first to match your workspace/layer name) to pull your own published features straight out of GeoServer as GeoJSON — no GeoServer preview page involved, just the raw WFS request.

</details>

In [ ]:
# Consume your own published WFS layer directly, the way any client would
# Edit WORKSPACE and LAYER_NAME to match what you published above.

import requests
import geopandas as gpd
from io import StringIO

WORKSPACE = "giss366"      # <-- change to your workspace name
LAYER_NAME = "your_layer"  # <-- change to your published layer name

wfs_url = (
    f"http://localhost:8080/geoserver/{WORKSPACE}/ows"
    f"?service=WFS&version=2.0.0&request=GetFeature"
    f"&typeName={WORKSPACE}:{LAYER_NAME}&outputFormat=application/json"
)

try:
    resp = requests.get(wfs_url, timeout=10)
    resp.raise_for_status()
    gdf = gpd.read_file(StringIO(resp.text))
    print(f"Pulled {len(gdf)} features straight from your own WFS endpoint.")
    display(gdf.head())
    gdf.plot(figsize=(6, 6))
except Exception as e:
    print("Couldn't reach the WFS endpoint yet -- confirm GeoServer is running and the")
    print("workspace/layer names above match what you published, then re-run this cell.")
    print(f"Error detail: {e}")

<details>
<summary><b>Proprietary pathway — consume it from ArcGIS Online: click to expand</b></summary>

1. In ArcGIS Online, open Map Viewer and choose **Add → Add Layer from Web**.
2. Choose **WMS OGC Web Service** (or **WFS**, if you'd rather bring in features instead of an image), and paste in your GeoServer WMS/WFS `GetCapabilities` URL from Step 4.
3. **This will likely fail** — and that's expected, not a mistake on your part. AGOL is a cloud service; it can only reach a server that's actually reachable from the open internet. A GeoServer running on `localhost` on your own laptop is invisible to anything outside your machine.

**Try one of these to actually get it working:**
- Use a tunneling tool like [ngrok](https://ngrok.com/) (free tier is fine) to temporarily expose your local GeoServer at a public URL: `ngrok http 8080`, then use the `https://...ngrok-free.app/geoserver/...` URL AGOL gives you instead of `localhost`.
- If your instructor has set up a shared, always-on GeoServer instance for the class, use that URL instead of your own local one.

Once AGOL successfully reads your `GetCapabilities` document, it will list your published layer(s) — add one to the map.

</details>

📸 **Save a screenshot now** — either your layer successfully added in ArcGIS Online, **or** the error message AGOL gave you if you weren't able to expose your local server. Both are valid, informative screenshots — document whichever one you actually got. Label it *Screenshot 4*.

---

## Step 5: Act

### What to turn in (graded — Lab Exercises)

1. Your four screenshots:
   - **Screenshot 1** — GeoServer welcome page (Collect)
   - **Screenshot 2** — raw `GetCapabilities` XML (Visualize)
   - **Screenshot 3** — your own layer in GeoServer's OpenLayers preview (Create)
   - **Screenshot 4** — your layer in ArcGIS Online, or the reachability error (Create)
2. Your completed Python cell output (the pulled GeoJSON feature count and the small plot).
3. A short written reflection (4-6 sentences) answering:
   - Walk through, in your own words, what happened between your browser (or AGOL, or the Python cell) and GeoServer for **one** of your requests today — what was requested, and what came back?
   - Revisit **EQ2**: what did publishing the *same* dataset as both WMS and WFS actually let you do differently between the two? Which one would you reach for if you needed to symbolize the data yourself in a client, versus just needing a quick reference layer?
   - If you weren't able to connect ArcGIS Online to your local GeoServer, explain *why* in your own words — this is itself the point of the exercise, not a failure to work around.
   - What's one thing that didn't work smoothly today, and what was your first troubleshooting move — docs, a web search of the exact error, or the course discussion board *(K16)*?

### Project Topic Ideation A (checkpoint — low-stakes, not graded on idea quality)

In 2-3 sentences each:
- What place are you drawn to for your final project? It doesn't need to be Grant County or anything geospatial you've worked with in this course — it can be anywhere that matters to you.
- Who's the audience for a web map of that place — and who is deliberately *not* the audience?
- What's one question that audience might actually want a map to answer for them?

You'll revisit this in **Topic Ideation B** (Week 5) once you've got more of the toolkit under your belt.

Post your screenshots, code output, and reflection to this week's submission space in Canvas.

### Looking Ahead

Week 3 shifts fully to the proprietary pathway — publishing a hosted feature layer and building a full web map in ArcGIS Online. Week 6 comes back to the tiling and cloud-native formats we flagged in lecture — worth re-reading your EQ2 answer above once you get there, to see if a static, serverless approach changes what you think the tradeoffs are.

---
### Resources

- GeoServer documentation: https://docs.geoserver.org/latest/en/user/
- GeoServer download: https://geoserver.org/download/
- NM RGIS Clearinghouse: https://rgis.unm.edu/
- OGC WMS standard: https://www.ogc.org/standard/wms/
- OGC WFS standard: https://www.ogc.org/standard/wfs/
- ngrok (tunneling a local server): https://ngrok.com/
- `geopandas` documentation: https://geopandas.org/